# FXS / 03 — Account growth trading every frozen structure (sizing only, no gates)

Question (Iván, 2026-11-09): how does an account grow trading **all** the intraday structures found in FXS 01, with position size increasing as equity grows?

**What is simulated.** Every frozen candidate from the 15 pair folders, traded every day exactly as frozen (time, direction), at quoted bid/ask, minus 0.70 bp commission. Each day every structure trades `floor(equity × w / 1,000) × 1,000` units, so size compounds with equity.

| Window | Role |
|---|---|
| **2021–2025** | **out of sample**: the structures were chosen without these years. This is the result to read. |
| 2016–2020 | in sample: the same years that selected the structures. Shown for contrast only; it is biased upward by construction. |

**Sizing schemes**
- **A. Fixed leverage per structure** L = 1, 2, 5, 10 (notional = L × equity on each structure).
- **B. Half-Kelly**, estimated on 2016–2020 only and applied unchanged to 2021–2025 (the honest ex-ante version), uncapped and capped at 30× total.
- **C. Hindsight**, only the structures that were positive after spread in 2021–2025. **Uses confirmation results to choose: not a valid test**, shown as an upper bound.

FXS 02's verdicts (0 of 10 confirmed) are unchanged by this notebook. `FXS_SYNTHETIC=1` runs offline on two synthetic structures.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

E0 = 10_000.0              # starting equity, USD
LEVERAGES = [1, 2, 5, 10]  # scheme A: notional per structure as a multiple of equity
EU_CAP = 30.0              # scheme B cap on total notional (EU retail FX limit)

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs03_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
BRANCH = Path.cwd().parent
OUT = Path.cwd() / ("03_synthetic" if SYNTHETIC else "03"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220, "display.max_columns", 40)

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for p, drift in (("EURUSD", 4.0), ("USDJPY", 0.0)):
        for a, b in [(dt.date(2015, 12, 28), dt.date(2020, 12, 31)), (dt.date(2020, 12, 28), dt.date(2025, 12, 31))]:
            F.write_synthetic_candles(base, p, a, b, seed=a.year + len(p), drift_bps=drift, price=110.0 if p.endswith("JPY") else 1.10)
    cand = pd.DataFrame([{"id": "EURUSD-01", "pair": "EURUSD", "clock": "NY", "start": 900, "end": 960, "direction": 1,
                          "start_time": "NY 08:00", "end_time": "NY 09:00"},
                         {"id": "USDJPY-01", "pair": "USDJPY", "clock": "UTC", "start": 60, "end": 120, "direction": 1,
                          "start_time": "UTC 01:00", "end_time": "UTC 02:00"}])
else:
    cand = F.load_frozen_candidates(BRANCH)

## 1. The structures

In [ ]:
display(cand[["id", "clock", "start_time", "end_time", "direction"] + [c for c in ("mean_bps", "t") if c in cand]].round(3))
conc = F.max_concurrent(cand)
print(f"{len(cand)} structures; at most {conc} open at the same minute")

## 2. Daily net returns per structure

Net = after the quoted spread and 0.70 bp commission, in bps of notional, one trade per structure per day.

In [ ]:
R_in = F.structure_returns(cand, *F.DISCOVERY, "discovery", cfg)
R_out = F.structure_returns(cand, *F.CONFIRMATION, "confirmation", cfg)
tab = pd.DataFrame({"in-sample mean bps": R_in.mean(), "in-sample total bps": R_in.sum(),
                    "OUT-OF-SAMPLE mean bps": R_out.mean(), "OUT-OF-SAMPLE total bps": R_out.sum(),
                    "out-of-sample days": (R_out != 0).sum()})
display(tab.round(2)); tab.to_csv(OUT / "03_structure_returns.csv")
print(f"equal-weight portfolio, mean net per day: in-sample {R_in.mean(axis=1).mean():.3f} bp, out-of-sample {R_out.mean(axis=1).mean():.3f} bp")
if R_out.shape[1] > 1:
    corr = R_out.corr()
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index)
    ax.set_title("2021-2025 daily return correlation"); fig.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout(); plt.savefig(OUT / "03_correlation.png", dpi=110); plt.show()

## 3. Scheme A: fixed leverage per structure, compounding

Each structure gets L × equity of notional every day. With all structures open at once the total is up to (number open) × L.

In [ ]:
rows, curves = [], {}
for window, R in (("2021-2025 out of sample", R_out), ("2016-2020 in sample", R_in)):
    for L in LEVERAGES:
        acc = F.simulate_account(R, pd.Series(float(L), index=R.columns), E0)
        curves[(window, L)] = acc
        rows.append({"window": window, "scheme": f"A: {L}x each", "max_total_notional_x": L * conc, **F.account_stats(acc, E0)})
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
for ax, window in zip(axes, ("2021-2025 out of sample", "2016-2020 in sample")):
    for L in LEVERAGES:
        ax.plot(curves[(window, L)]["equity"].clip(lower=1), label=f"{L}x each")
    ax.axhline(E0, color="grey", lw=0.6); ax.set_yscale("log"); ax.set_title(f"Scheme A, {window}"); ax.legend()
axes[0].set_ylabel("equity, USD (log)")
plt.tight_layout(); plt.savefig(OUT / "03_scheme_A.png", dpi=110); plt.show()

## 4. Scheme B: half-Kelly estimated on 2016–2020, applied to 2021–2025

The sizes are decided before seeing 2021–2025, as they would have been in real trading. Structures with a negative Kelly weight get zero.

In [ ]:
wB = F.kelly_weights(R_in, 0.5)
wBc = F.kelly_weights(R_in, 0.5, cap_total=EU_CAP)
display(pd.DataFrame({"half-Kelly notional x equity": wB, f"capped at {EU_CAP:.0f}x total": wBc}).round(2))
for name, w in (("B: half-Kelly (2016-2020 estimate)", wB), (f"B: half-Kelly capped {EU_CAP:.0f}x", wBc)):
    for window, R in (("2021-2025 out of sample", R_out), ("2016-2020 in sample", R_in)):
        acc = F.simulate_account(R, w, E0)
        curves[(window, name)] = acc
        rows.append({"window": window, "scheme": name, "max_total_notional_x": float(w.sum()), **F.account_stats(acc, E0)})
fig, ax = plt.subplots(figsize=(11, 4))
for name in ("B: half-Kelly (2016-2020 estimate)", f"B: half-Kelly capped {EU_CAP:.0f}x"):
    ax.plot(curves[("2021-2025 out of sample", name)]["equity"].clip(lower=1), label=name)
ax.axhline(E0, color="grey", lw=0.6); ax.set_yscale("log"); ax.legend(); ax.set_title("Scheme B on 2021-2025 (out of sample)")
plt.tight_layout(); plt.savefig(OUT / "03_scheme_B.png", dpi=110); plt.show()

## 5. Scheme C: hindsight (NOT a valid test)

Only the structures whose 2021–2025 mean after spread was positive and significant (K1). Choosing them uses the results being measured, so this is an upper bound, not an expectation.

In [ ]:
conf = []
for p in sorted(set(cand["pair"])):
    f = BRANCH / p / "02" / "02_confirmation.csv"
    if f.exists():
        conf.append(pd.read_csv(f))
hind = (pd.concat(conf).query("K1 == True")["id"].tolist() if conf else [])
if SYNTHETIC:
    hind = [c for c in R_out.columns if R_out[c].mean() > 0]
print("hindsight set:", hind)
if hind:
    for L in LEVERAGES:
        acc = F.simulate_account(R_out[hind], pd.Series(float(L), index=hind), E0)
        curves[("2021-2025 out of sample", f"C{L}")] = acc
        rows.append({"window": "2021-2025 out of sample", "scheme": f"C (hindsight): {L}x each", "max_total_notional_x": L * len(hind),
                     **F.account_stats(acc, E0)})

## 6. Results

In [ ]:
res = pd.DataFrame(rows)
show = res.copy()
for c in ("cagr", "ann_vol", "max_drawdown", "worst_year", "best_year"):
    show[c] = (100 * show[c]).round(1).astype(str) + "%"
show["final_equity"] = show["final_equity"].round(0)
display(show.sort_values(["window", "scheme"]).reset_index(drop=True))
res.to_csv(OUT / "03_results.csv", index=False)

## 7. Where the out-of-sample P&L came from

Scheme A at 5× each, 2021–2025: dollar P&L per structure (sizes compound with the whole account).

In [ ]:
L = 5
acc = curves[("2021-2025 out of sample", L)]
eq_prev = acc["equity"].shift(1).fillna(E0)
units = np.floor(eq_prev.to_numpy()[:, None] * L / 1000) * 1000
pnl = pd.DataFrame(units * R_out.to_numpy() / 1e4, index=R_out.index, columns=R_out.columns)
contrib = pnl.sum().sort_values()
display(contrib.round(0).to_frame("USD P&L 2021-2025"))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pnl.cumsum()); ax.axhline(0, color="black", lw=0.5); ax.legend(pnl.columns, ncol=4, fontsize=8)
ax.set_title(f"cumulative USD P&L by structure, {L}x each, 2021-2025"); plt.tight_layout(); plt.savefig(OUT / "03_contribution.png", dpi=110); plt.show()

In [ ]:
summary = {"study": "FXS03", "fxs_version": F.FXS_VERSION, "synthetic": SYNTHETIC, "equity0": E0,
           "structures": cand["id"].tolist(), "max_concurrent": conc,
           "equal_weight_mean_net_bps": {"in_sample": float(R_in.mean(axis=1).mean()), "out_of_sample": float(R_out.mean(axis=1).mean())},
           "kelly_weights_half": wB.round(3).to_dict(), "hindsight_set": hind, "results": res.to_dict(orient="records"),
           "note": "Sizing only. FXS 02 verdicts unchanged; scheme C uses confirmation results to choose and is not a test."}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("structures", "max_concurrent", "equal_weight_mean_net_bps")}, indent=2))